# Infraestructuras Computacionales para el Procesamiento de datos masivos

**Ejercicio 1, Módulo 3 Tema 1 — Consumidor de streaming con Kafka**

Este cuaderno consume eventos JSON desde un tópico de **Kafka** y calcula métricas con **Spark Structured Streaming**.

En esta versión, Kafka se utiliza a nivel básico como intermediario entre el productor y Spark:

```text
CoinGecko API
     ↓
Productor Python
     ↓
Kafka topic: crypto_prices
     ↓
Spark Structured Streaming
     ↓
Agregaciones por ventana
     ↓
Consola / Parquet
```

El objetivo no es profundizar todavía en Kafka, sino entender cómo Spark puede leer un flujo de eventos desde un sistema de mensajería.

## 1. Preparación del entorno

Ejecuta esta celda para crear las carpetas necesarias para la salida en Parquet y los checkpoints.

> Importante: ejecuta primero el cuaderno productor. Ese cuaderno arranca Kafka, crea el tópico `crypto_prices` y publica eventos en `localhost:9092`.

In [ ]:
import os
import shutil

OUTPUT_DIR = os.path.abspath("output/crypto_prices_parquet")
CHECKPOINT_DIR = os.path.abspath("checkpoint/crypto_prices_kafka")

# Limpieza para repetir el ejercicio desde cero.
# No se elimina ningún dato de Kafka; solo la salida y checkpoints de Spark.
for path in (OUTPUT_DIR, CHECKPOINT_DIR):
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)

print("Directorio de trabajo:", os.getcwd())
print("Carpetas preparadas y limpiadas:")
print("-", OUTPUT_DIR)
print("-", CHECKPOINT_DIR)

## 2. Instalación de PySpark

Si PySpark ya está instalado en tu entorno, puedes omitir esta celda.

Para leer desde Kafka, Spark necesita además el paquete `spark-sql-kafka-0-10`. Lo añadiremos al crear la `SparkSession`.

In [ ]:
# Ejecutar solo si PySpark no está instalado en el entorno
%pip install pyspark==3.5.1

## 3. Inicialización de SparkSession con soporte Kafka

Spark no incluye el conector Kafka en el paquete base de PySpark. Por eso se añade mediante `spark.jars.packages`.

La versión del conector debe coincidir con la versión de Spark usada en el entorno. En este ejemplo usamos Spark 3.5.1.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CoinGeckoKafkaStreaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

spark

## 4. Configuración básica de Kafka

En este ejercicio se usará un único tópico: `crypto_prices`.

Cada mensaje Kafka debe tener en el campo `value` un JSON con esta estructura:

```json
{
  "event_time": "2026-04-27T10:30:00Z",
  "coin": "bitcoin",
  "currency": "eur",
  "price": 58750.23
}
```

In [ ]:
KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "crypto_prices"

print("Bootstrap servers:", KAFKA_BOOTSTRAP_SERVERS)
print("Topic:", KAFKA_TOPIC)

## 5. Definición del esquema de los eventos

Kafka entrega los mensajes como bytes. Spark leerá primero el `value` como texto y después lo transformará a columnas usando este esquema.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("coin", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("price", DoubleType(), True)
])

schema

## 6. Lectura del flujo desde Kafka

Spark leerá de Kafka mediante `readStream`.

Columnas importantes que entrega Kafka:

- `key`: clave del mensaje, si existe.
- `value`: contenido del mensaje.
- `topic`: tópico de origen.
- `partition`: partición Kafka.
- `offset`: posición del mensaje dentro de la partición.
- `timestamp`: marca temporal asignada por Kafka.

En este ejercicio nos centraremos en `value`, que contiene el JSON de CoinGecko.

In [ ]:
raw_kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

raw_kafka_df.printSchema()

## 7. Parseo del mensaje JSON

El campo `value` llega en binario, así que se convierte a `STRING` y después se aplica `from_json()`.

Además, conservamos metadatos básicos de Kafka como `topic`, `partition`, `offset` y `kafka_timestamp`.

In [ ]:
from pyspark.sql.functions import col, from_json

stream_df = (
    raw_kafka_df
    .select(
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp"),
        col("value").cast("string").alias("json_value")
    )
    .select(
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        from_json(col("json_value"), schema).alias("data")
    )
    .select(
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        col("data.event_time").alias("event_time"),
        col("data.coin").alias("coin"),
        col("data.currency").alias("currency"),
        col("data.price").alias("price")
    )
)

stream_df.printSchema()

## 8. Consulta básica en consola

Esta consulta permite comprobar que Spark está recibiendo mensajes desde Kafka.

Ejecuta primero el cuaderno productor y después esta celda. Con `startingOffsets="earliest"`, Spark puede leer también los eventos que ya estaban en el tópico antes de arrancar esta consulta.

In [ ]:
raw_query = (
    stream_df.writeStream
    .format("console")
    .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "raw_console"))
    .outputMode("append")
    .option("truncate", "false")
    .trigger(processingTime="10 seconds")
    .start()
)

raw_query.awaitTermination(60)
raw_query.stop()

## 9. Agregaciones por ventana temporal

Ahora se calculan métricas por criptomoneda en ventanas de 1 minuto:

- precio medio,
- precio máximo,
- precio mínimo,
- número de eventos procesados.

La marca de agua (`watermark`) permite gestionar eventos que lleguen tarde según el campo `event_time`.

In [ ]:
from pyspark.sql.functions import window, avg, max, min, count

metrics_df = (
    stream_df
    .withWatermark("event_time", "1 minute")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("coin")
    )
    .agg(
        avg("price").alias("avg_price"),
        max("price").alias("max_price"),
        min("price").alias("min_price"),
        count("*").alias("num_events")
    )
)

metrics_df.printSchema()

## 10. Visualización de métricas en consola

La salida en modo `update` muestra las ventanas que se actualizan cuando llegan nuevos eventos.

De nuevo, la consulta se detiene automáticamente después de 60 segundos para que el ejercicio sea cómodo en Jupyter.

In [ ]:
metrics_query = (
    metrics_df.writeStream
    .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "metrics_console"))
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="10 seconds")
    .start()
)

metrics_query.awaitTermination(60)
metrics_query.stop()

## 11. Escritura de eventos en Parquet

Esta consulta guarda los eventos ya parseados en formato Parquet.

En streaming, Spark necesita una carpeta de checkpoint para recordar qué offsets de Kafka ya ha procesado.

In [ ]:
parquet_query = (
    stream_df.writeStream
    .format("parquet")
    .option("path", OUTPUT_DIR)
    .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "parquet"))
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start()
)

parquet_query.awaitTermination(60)
parquet_query.stop()

## 12. Lectura posterior de los datos Parquet

Después de que la consulta anterior haya procesado algunos eventos, los datos almacenados se pueden leer como un DataFrame estático.

In [ ]:
historical_df = spark.read.parquet(OUTPUT_DIR)

historical_df.show(truncate=False)
historical_df.groupBy("coin").count().show()

## 13. Detener consultas activas

Es importante detener las consultas streaming cuando se termina el ejercicio para liberar recursos.

In [ ]:
for query in spark.streams.active:
    print("Deteniendo consulta:", query.name, query.id)
    query.stop()

print("Consultas activas:", spark.streams.active)

## 14. Cierre de SparkSession

Ejecuta esta celda al final del ejercicio.

In [ ]:
spark.stop()

## 15. Preguntas de autoevaluación

1. ¿Qué diferencia hay entre leer ficheros JSON como streaming y leer eventos desde Kafka?
2. ¿Qué papel cumple Kafka entre el productor y Spark?
3. ¿Por qué Spark recibe el campo `value` como binario?
4. ¿Para qué sirven `partition` y `offset`?
5. ¿Qué significa `startingOffsets="latest"`?
6. ¿Qué ocurriría si se usa `startingOffsets="earliest"`?
7. ¿Por qué el checkpoint es especialmente importante al consumir desde Kafka?
8. ¿Qué diferencia hay entre `event_time` y `kafka_timestamp`?
9. ¿Qué ventajas tiene Kafka frente a una carpeta de ficheros como fuente streaming?
10. ¿Qué limitaciones tiene este ejemplo respecto a una arquitectura de producción?

## 16. Ejercicios de ampliación

1. Cambiar `startingOffsets` de `latest` a `earliest` y observar el resultado.
2. Añadir una columna calculada con el precio redondeado a dos decimales.
3. Filtrar solo los eventos de `bitcoin`.
4. Calcular ventanas de 5 minutos en lugar de 1 minuto.
5. Guardar las métricas agregadas en Parquet.
6. Crear una alerta cuando el precio de Bitcoin supere un umbral definido.
7. Comparar `event_time` con `kafka_timestamp`.
8. Modificar el productor para enviar la criptomoneda como `key` del mensaje Kafka.